# 00 — Núcleo compartilhado (POC de validação de construto e calibração)

**O que faz:** carrega `config`, copia (sem reimplementar) os descritores, regras e
motor de avaliação do notebook `heuristics_activation_pipeline_v2.ipynb` (conteúdo v3),
constrói e cacheia em disco `interp_cache`, `pair_cache`, `pair_cache_self` e `df_sector`,
e define o mapeamento regra → comportamento (`RULE_BEHAVIOR`) usado pelo instrumento de
rotulagem.

**Consome:** `config/` (pacote), `.ibt` originais em
`G:/Meu Drive/Estudos/Datasets - Simracing/...` (via `DATASETS`/`resolve_stint_files`),
`img/batch_heuristics_v3/heuristic_activations_long.csv` (para o sanity check final).

**Produz:** `data/poc_threshold_validation/cache/interp_cache.pkl`,
`cache/pair_cache.pkl`, `cache/pair_cache_self.pkl`, `cache/df_sector.parquet`.
Este notebook é carregado pelos demais via `%run 00_core.ipynb` — não é executado
diretamente por eles de outra forma.

**Ordem de execução:** é o primeiro notebook (00), roda uma única vez para construir o
cache; execuções seguintes de `01`-`04` reutilizam o cache em disco sem tocar nos `.ibt`.

**Nota sobre esta execução:** por decisão do usuário, a carga completa de telemetria
(~54 arquivos `.ibt`, 6 pilotos × 2 pistas) **não foi executada nesta sessão** (10-20 min
estimados). A célula "SMOKE TEST" abaixo valida o pipeline inteiro com 3 arquivos reais
(`Tomaz/stint_ref`, `Tomaz/stint_1`, `Morsinaldo/stint_1` em `charlotte_roval_2025`, já
testados). A célula "BUILD COMPLETO" está pronta e correta, mas fica para o usuário rodar
(`RUN_FULL_BUILD = True`) fora desta sessão.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from scipy.stats import kendalltau

# notebook lives in Iracing/Notebooks/poc_threshold_validation/ -> go up two levels
PROJECT_ROOT = Path("..").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import irsdk  # noqa: F401
except ImportError as e:
    raise ImportError(
        "Modulo 'irsdk' ausente. O pacote no PyPI chama-se 'pyirsdk' "
        "(nao 'irsdk'): rode `pip install pyirsdk`."
    ) from e

from config import (TRACK_CONFIGS, DATASETS, DRIVER_ALIAS, resolve_stint_files,
                    basic_clean_and_units, build_lap_validity_table, load_stint)

RNG = np.random.default_rng(20250813)

DATA_DIR    = PROJECT_ROOT / "data" / "poc_threshold_validation"
CACHE_DIR   = DATA_DIR / "cache"
RESULTS_DIR = DATA_DIR / "results"
PANELS_DIR  = DATA_DIR / "panels"
for d in (CACHE_DIR, RESULTS_DIR, PANELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"Tracks       : {list(TRACK_CONFIGS.keys())}")
print(f"Drivers      : {DRIVER_ALIAS}")

PROJECT_ROOT : C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing
DATA_DIR     : C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation
Tracks       : ['charlotte_roval_2025', 'summit_point']
Drivers      : {'Rodrigo': 'Driver A', 'Tomaz': 'Driver B', 'Morsinaldo': 'Driver C', 'Thallys': 'Driver D', 'Igor': 'Driver E', 'Hilton': 'Driver F'}


## Definições copiadas de `heuristics_activation_pipeline_v2.ipynb` (v3)

As células a seguir (configuração de comparações, descritores, `DRIVING_RULES`,
`THRESHOLD_SPEC`, motor de cache de dois estágios) são **copiadas literalmente** do
notebook v3 — fonte da verdade dos descritores e regras. Nenhuma lógica de descritor ou
de regra é reimplementada aqui; apenas o `SAVE_DIR` e a adição de `pair_cache_self` são
específicos desta POC.

In [2]:
COMPARISONS = [
    {"ref": "Rodrigo",     "ref_stint": "stint_1",   "test": "Tomaz",      "label": "B vs. A"},
    {"ref": "Tomaz",       "ref_stint": "stint_ref", "test": "Morsinaldo", "label": "C vs. B"},
    {"ref": "Tomaz",       "ref_stint": "stint_ref", "test": "Thallys",    "label": "D vs. B"},
    {"ref": "Tomaz",       "ref_stint": "stint_ref", "test": "Igor",       "label": "E vs. B"},
    {"ref": "Tomaz",       "ref_stint": "stint_ref", "test": "Hilton",     "label": "F vs. B"},
]
comp_order = [c["label"] for c in COMPARISONS]

EXCLUDE_STINTS = {"warmup", "stint_ref"}
TRACK_SHORT    = {"charlotte_roval_2025": "CLT", "summit_point": "SPT"}
BASE_GRID_LEN  = 2000

# Longitudinal design: pairs that received feedback vs. the untreated control.
FEEDBACK_PAIRS = ["B vs. A", "C vs. B", "D vs. B", "F vs. B"]
CONTROL_PAIR   = "E vs. B"

SELF = {"ref": "Tomaz", "ref_stint": "stint_ref", "test": "Tomaz",
        "label": "B vs. B (self)"}

print(f"{len(COMPARISONS)} comparisons: {comp_order}")
print(f"Self-comparison label: {SELF['label']!r}")

5 comparisons: ['B vs. A', 'C vs. B', 'D vs. B', 'E vs. B', 'F vs. B']
Self-comparison label: 'B vs. B (self)'


In [3]:
# --- Descriptor-level parameters (copied from v3) -----------------------------
DESC_SPEC = {
    "PEDAL_ACTIVE_PCT": 5.0,   # %  -- pedal considered engaged (masks M_b, M_t)
    "MIN_BRAKE_RUN":    5,     # consecutive samples on the 2000-pt/lap grid
}

DESC_LEGACY = {
    "PEDAL_ACTIVE_PCT": 0.05,
    "MIN_BRAKE_RUN":    5,
}

EPS = 1e-6


def compute_derivatives(sec):
    """Pedal rates in %/s, Savitzky-Golay smoothed (window <= 9, order 3)."""
    t  = sec["t_rel"]
    dt = np.mean(np.diff(t)) if len(t) > 1 else 0.01
    brake_rate    = np.gradient(sec["brake"], dt)
    throttle_rate = np.gradient(sec["throttle"], dt)
    window = min(9, len(brake_rate))
    if window > 3 and window % 2 != 0:
        brake_rate    = savgol_filter(brake_rate, window, 3)
        throttle_rate = savgol_filter(throttle_rate, window, 3)
    return brake_rate, throttle_rate


def braking_heuristics(sec, brake_rate, dp):
    d = {}
    m = sec["brake"] > dp["PEDAL_ACTIVE_PCT"]
    if np.any(m):
        d["BrakeRampRate"]    = float(np.quantile(brake_rate[m], 0.95))
        d["BrakeConsistency"] = float(np.var(brake_rate[m]))
        d["BrakeEfficiency"]  = float(abs(np.min(sec["long_accel"][m])) /
                                      (np.max(sec["brake"][m]) + EPS))
    else:
        d["BrakeRampRate"] = d["BrakeConsistency"] = d["BrakeEfficiency"] = 0.0
    return d


def brake_start_dist(sec, dp):
    n  = int(dp["MIN_BRAKE_RUN"])
    on = sec["brake"] > dp["PEDAL_ACTIVE_PCT"]
    if on.size < n or not np.any(on):
        return np.nan
    runs = np.convolve(on.astype(int), np.ones(n, dtype=int), mode="valid")
    idx  = np.flatnonzero(runs == n)
    return float(sec["lap_pct"][idx[0]]) if idx.size else np.nan


def rotation_heuristics(sec):
    d = {}
    speed, yaw, steer, t = sec["speed"], sec["yaw_rate"], sec["steering"], sec["t_rel"]
    idx_vmin = np.argmin(speed)
    idx_mrp  = np.argmax(np.abs(yaw))
    d["Vmin"] = float(speed[idx_vmin])
    d["Delta_MRP_Vmin"] = float(t[idx_vmin] - t[idx_mrp])
    dt = np.mean(np.diff(t)) if len(t) > 1 else 0.01
    d["TrailOverlap"] = float((sec["brake"] * np.abs(steer) * dt).sum())
    d["Vmin_Norm"]    = float(d["Vmin"] / (speed[0] + EPS))
    avg_steer = np.mean(np.abs(steer))
    d["RotationEfficiency"] = (float(np.mean(np.abs(yaw)) / (avg_steer + EPS))
                               if avg_steer > 1e-3 else 0.0)
    return d


def acceleration_heuristics(sec, throttle_rate, dp):
    d = {}
    m = sec["throttle"] > dp["PEDAL_ACTIVE_PCT"]
    if np.any(m):
        d["ThrottleAttackRate"]    = float(np.quantile(throttle_rate[m], 0.95))
        d["ThrottleSmoothness"]    = float(1.0 / (np.var(throttle_rate[m]) + EPS))
        d["ThrottleSteerConflict"] = float(np.mean(sec["throttle"] *
                                                   np.abs(sec["steering"])))
        d["AccelEfficiency"]       = float(np.mean(sec["long_accel"][m]) /
                                           (np.mean(sec["throttle"][m]) + EPS))
    else:
        d["ThrottleAttackRate"] = d["ThrottleSmoothness"] = 0.0
        d["ThrottleSteerConflict"] = d["AccelEfficiency"] = 0.0
    return d


def sector_heuristics(sec_raw, dp):
    sec = {
        "t_rel":      sec_raw["t_rel"],
        "speed":      sec_raw["speed"],
        "brake":      sec_raw["brake"],
        "throttle":   sec_raw["throttle"],
        "steering":   sec_raw["SteeringWheelAngle"],
        "yaw_rate":   sec_raw["YawRate"],
        "long_accel": sec_raw.get("LongAccel", np.zeros_like(sec_raw["speed"])),
        "lap_pct":    sec_raw["LapDistPct"],
    }
    brake_rate, throttle_rate = compute_derivatives(sec)
    h = {
        "SpeedStart":     float(sec["speed"][0]),
        "SpeedEnd":       float(sec["speed"][-1]),
        "BrakePeak":      float(np.max(sec["brake"])),
        "SteerRMS":       float(np.sqrt(np.mean(sec["steering"] ** 2))),
        "BrakeStartDist": brake_start_dist(sec, dp),
    }
    h.update(braking_heuristics(sec, brake_rate, dp))
    h.update(rotation_heuristics(sec))
    h.update(acceleration_heuristics(sec, throttle_rate, dp))
    return h


def compute_sector_heuristics(interp, edges, dp):
    if not interp:
        return pd.DataFrame()
    lap_dist = interp["LapDistPct"]
    rows = []
    for i in range(len(edges) - 1):
        mask = (lap_dist >= edges[i]) & (lap_dist < edges[i + 1])
        if i == len(edges) - 2:
            mask = (lap_dist >= edges[i]) & (lap_dist <= edges[i + 1])
        idx = np.flatnonzero(mask)
        if idx.size == 0:
            continue
        sl = slice(idx[0], idx[-1] + 1)
        sec_raw = {k: v[sl] for k, v in interp.items()}
        sec_raw["t_rel"] = sec_raw["t_rel"] - sec_raw["t_rel"][0]
        h = sector_heuristics(sec_raw, dp)
        h["Sector"]       = i
        h["SectorTime_s"] = float(sec_raw["t_rel"][-1])
        rows.append(h)
    return pd.DataFrame(rows)


print(f"Descriptor functions loaded. DESC_SPEC = {DESC_SPEC}")

Descriptor functions loaded. DESC_SPEC = {'PEDAL_ACTIVE_PCT': 5.0, 'MIN_BRAKE_RUN': 5}


In [4]:
# --- Lap alignment (copied from v3) --------------------------------------------
def align_lap_by_dist(g: pd.DataFrame, grid: np.ndarray):
    g = g.sort_values("LapDistPct").drop_duplicates(subset=["LapDistPct"], keep="first")
    if g.empty:
        return {}
    t_rel = g["SessionTime"] - g["SessionTime"].iloc[0]
    x = g["LapDistPct"].to_numpy()
    if len(x) < 2 or np.allclose(x.max() - x.min(), 0):
        return {}
    interp = lambda y: np.interp(grid, x, y)
    return {
        "LapDistPct": grid,
        "t_rel":      interp(t_rel.to_numpy()),
        "speed":      interp(g["Speed_KPH"].to_numpy()),
        "throttle":   interp(g["Throttle_Pct"].to_numpy()),
        "brake":      interp(g["Brake_Pct"].to_numpy()),
        "SteeringWheelAngle": interp(g["SteeringWheelAngle"].to_numpy()),
        "YawRate":    interp(g.get("YawRate",   pd.Series(np.zeros_like(x))).to_numpy()),
        "LongAccel":  interp(g.get("LongAccel", pd.Series(np.zeros_like(x))).to_numpy()),
    }


def assert_pedal_scale(interp, name="lap"):
    """Fail loudly if the pedal channels are not on a 0-100 scale."""
    for ch in ("brake", "throttle"):
        vmax = float(np.max(interp[ch]))
        if vmax <= 1.5:
            raise ValueError(
                f"[{name}] channel '{ch}' peaks at {vmax:.3f}: looks like a 0-1 scale. "
                f"All thresholds assume 0-100. Fix basic_clean_and_units() or rescale.")
    return True


print("align_lap_by_dist / assert_pedal_scale loaded.")

align_lap_by_dist / assert_pedal_scale loaded.


In [5]:
# --- Thresholds and rules (copied from v3) -------------------------------------
THRESHOLD_SPEC = {
    'MRP_DT':            (0.0,  -0.05),
    'BRAKE_EFF_RATIO':   (1.0,  -0.15),
    'TRAIL_RATIO':       (1.0,  -0.30),
    'LEGACY_DV':         (0.0,  -2.0),
    'LEGACY_BRAKE_GATE': (0.0,   5.0),
    'BRAKE_DIST_TOL':    (0.0,   0.005),
    'OVERBRAKE_DELTA':   (0.0,  15.0),
    'ENTRY_SLOW_REL':    (0.0,  -0.015),
    'ENTRY_BRAKE_GATE':  (0.0,   5.0),
    'RAMP_RATIO':        (1.0,   0.25),
    'CONS_RATIO':        (1.0,   0.30),
    'STEER_RMS_RATIO':   (1.0,   0.20),
    'ROTEFF_DIFF':       (0.0,  -0.05),
    'VMIN_RATIO':        (1.0,  -0.03),
    'SMOOTH_RATIO':      (1.0,  -0.20),
    'EXIT_DV':           (0.0,  -1.0),
    'CONFLICT_RATIO':    (1.0,   0.20),
}

THRESHOLD_SPEC_LEGACY = dict(THRESHOLD_SPEC,
                             OVERBRAKE_DELTA=(0.0, 0.15),
                             LEGACY_BRAKE_GATE=(0.0, 0.05))


def thresholds_at(factors=None, spec=None):
    """Effective thresholds; `factors` maps parameter name -> scale factor."""
    spec = spec or THRESHOLD_SPEC
    factors = factors or {}
    return {k: n + m * factors.get(k, 1.0) for k, (n, m) in spec.items()}


TH = thresholds_at()

DRIVING_RULES = [
    {"id": "BRAKING_POINT_MISMATCH", "category": "Braking", "params": ["BRAKE_DIST_TOL"],
     "condition": lambda A, B: (np.isfinite(A.get("BrakeStartDist", np.nan)) and
                                np.isfinite(B.get("BrakeStartDist", np.nan)) and
                                abs(B["BrakeStartDist"] - A["BrakeStartDist"]) > TH['BRAKE_DIST_TOL'])},

    {"id": "OVER_BRAKING_PEAK", "category": "Braking", "params": ["OVERBRAKE_DELTA"],
     "condition": lambda A, B: B.get("BrakePeak", 0) > A.get("BrakePeak", 0) + TH['OVERBRAKE_DELTA']},

    {"id": "LOW_BRAKE_EFFICIENCY", "category": "Braking", "params": ["BRAKE_EFF_RATIO"],
     "condition": lambda A, B: B.get("BrakeEfficiency", 0) <
                               A.get("BrakeEfficiency", 0) * TH['BRAKE_EFF_RATIO']},

    {"id": "AGGRESSIVE_UNSTABLE_BRAKE", "category": "Braking", "params": ["RAMP_RATIO", "CONS_RATIO"],
     "condition": lambda A, B: ((B.get("BrakeRampRate", 1) / (A.get("BrakeRampRate", 1) + EPS)) > TH['RAMP_RATIO'] and
                                (B.get("BrakeConsistency", 1) / (A.get("BrakeConsistency", 1) + EPS)) > TH['CONS_RATIO'])},

    {"id": "ENTRY_OVER_SLOW", "category": "Braking", "params": ["ENTRY_SLOW_REL", "ENTRY_BRAKE_GATE"],
     "condition": lambda A, B: (((B.get("Vmin", 1) - A.get("Vmin", 1)) /
                                 (A.get("Vmin", 1) + EPS)) < TH['ENTRY_SLOW_REL'] and
                                B.get("BrakePeak", 0) > TH['ENTRY_BRAKE_GATE'])},

    {"id": "LATE_ROTATION", "category": "Rotation", "params": ["MRP_DT"],
     "condition": lambda A, B: B.get("Delta_MRP_Vmin", 0) < TH['MRP_DT']},

    {"id": "POOR_TRAIL_BRAKING", "category": "Rotation", "params": ["TRAIL_RATIO"],
     "condition": lambda A, B: (B.get("TrailOverlap", 0) < A.get("TrailOverlap", 0) * TH['TRAIL_RATIO'] and
                                B.get("Vmin", 0) < A.get("Vmin", 0))},

    {"id": "STEER_EFFICIENCY", "category": "Rotation", "params": ["STEER_RMS_RATIO"],
     "condition": lambda A, B: B.get("SteerRMS", 0) > A.get("SteerRMS", 0) * TH['STEER_RMS_RATIO']},

    {"id": "ROTATION_INEFFICIENT", "category": "Rotation", "params": ["ROTEFF_DIFF", "VMIN_RATIO"],
     "condition": lambda A, B: ((B.get("RotationEfficiency", 0) -
                                 A.get("RotationEfficiency", 0) < TH['ROTEFF_DIFF']) and
                                (B.get("Vmin", 0) < A.get("Vmin", 0) * TH['VMIN_RATIO']))},

    {"id": "THROTTLE_STEER_CONFLICT", "category": "Traction", "params": ["CONFLICT_RATIO"],
     "condition": lambda A, B: B.get("ThrottleSteerConflict", 0) >
                               A.get("ThrottleSteerConflict", 0) * TH['CONFLICT_RATIO']},

    {"id": "LATE_THROTTLE_LOW_SPEED", "category": "Traction", "params": ["SMOOTH_RATIO", "EXIT_DV"],
     "condition": lambda A, B: ((B.get("ThrottleSmoothness", 1) <
                                 A.get("ThrottleSmoothness", 1) * TH['SMOOTH_RATIO']) and
                                (B.get("SpeedEnd", 0) < A.get("SpeedEnd", 0) + TH['EXIT_DV']))},

    {"id": "EXIT_SPEED_LEGACY", "category": "Legacy", "params": ["LEGACY_DV", "LEGACY_BRAKE_GATE"],
     "condition": lambda A, B: (B.get("SpeedStartDiff", 0) < TH['LEGACY_DV'] and
                                B.get("BrakePeak", 0) < TH['LEGACY_BRAKE_GATE'])},
]

RULE_IDS   = [r["id"] for r in DRIVING_RULES]
RULE_CAT   = {r["id"]: r["category"] for r in DRIVING_RULES}
PARAM_RULES = {}
for r in DRIVING_RULES:
    for p in r["params"]:
        PARAM_RULES.setdefault(p, []).append(r["id"])

print(f"{len(DRIVING_RULES)} rules, {len(THRESHOLD_SPEC)} tunable thresholds, "
      f"{len(DESC_SPEC)} descriptor parameters.")
for k, v in thresholds_at().items():
    print(f"  {k:20s} = {v:>9.4g}")

12 rules, 17 tunable thresholds, 2 descriptor parameters.
  MRP_DT               =     -0.05
  BRAKE_EFF_RATIO      =      0.85
  TRAIL_RATIO          =       0.7
  LEGACY_DV            =        -2
  LEGACY_BRAKE_GATE    =         5
  BRAKE_DIST_TOL       =     0.005
  OVERBRAKE_DELTA      =        15
  ENTRY_SLOW_REL       =    -0.015
  ENTRY_BRAKE_GATE     =         5
  RAMP_RATIO           =      1.25
  CONS_RATIO           =       1.3
  STEER_RMS_RATIO      =       1.2
  ROTEFF_DIFF          =     -0.05
  VMIN_RATIO           =      0.97
  SMOOTH_RATIO         =       0.8
  EXIT_DV              =        -1
  CONFLICT_RATIO       =       1.2


## `RULE_BEHAVIOR` — mapeamento regra → item do checklist de rotulagem

Específico desta POC (não existe no v3). `None` marca regras que nunca entram na
rotulagem cega, apenas na validação de critério (item 2 de `01_gap_closure.ipynb`).

In [6]:
RULE_BEHAVIOR = {
    "BRAKING_POINT_MISMATCH":   "brake_point_differs",
    "OVER_BRAKING_PEAK":        "over_braking",
    "ENTRY_OVER_SLOW":          "entry_too_slow",
    "EXIT_SPEED_LEGACY":        "exit_slow_no_brake",
    "POOR_TRAIL_BRAKING":       "no_trail_braking",
    "LATE_ROTATION":            "rotation_after_vmin",
    "STEER_EFFICIENCY":         "excess_steering",
    "AGGRESSIVE_UNSTABLE_BRAKE":"brake_application_abrupt",
    "LATE_THROTTLE_LOW_SPEED":  "throttle_hesitation_slow_exit",
    "THROTTLE_STEER_CONFLICT":  "throttle_with_steering",
    "LOW_BRAKE_EFFICIENCY":     None,
    "ROTATION_INEFFICIENT":     None,
}

# Short Portuguese labels for the labeling UI (03) -- never show the Rule_ID itself.
BEHAVIOR_LABEL_PT = {
    "brake_point_differs":           "Ponto de frenagem diferente",
    "over_braking":                  "Frenagem mais forte que a referência",
    "entry_too_slow":                "Entrada mais lenta que a referência",
    "exit_slow_no_brake":            "Perda herdada: já entra mais devagar, sem frear mais",
    "no_trail_braking":              "Sem trail braking / solta o freio cedo",
    "rotation_after_vmin":           "Gira o carro depois do ponto mais lento",
    "excess_steering":               "Esterçamento excessivo",
    "brake_application_abrupt":      "Frenagem abrupta / instável",
    "throttle_hesitation_slow_exit": "Hesita no acelerador na saída lenta",
    "throttle_with_steering":        "Acelera enquanto ainda esterça bastante",
}

assert set(RULE_BEHAVIOR) == set(RULE_IDS), "RULE_BEHAVIOR must cover every DRIVING_RULES id"
behaviors_in_labeling = sorted({b for b in RULE_BEHAVIOR.values() if b is not None})
assert set(behaviors_in_labeling) == set(BEHAVIOR_LABEL_PT), \
    "BEHAVIOR_LABEL_PT must cover every non-None behavior in RULE_BEHAVIOR"

print(f"{len(RULE_BEHAVIOR)} rules mapped, {len(behaviors_in_labeling)} enter labeling, "
      f"{sum(v is None for v in RULE_BEHAVIOR.values())} criterion-only.")
for rid, beh in RULE_BEHAVIOR.items():
    print(f"  {rid:28s} -> {beh}")

12 rules mapped, 10 enter labeling, 2 criterion-only.
  BRAKING_POINT_MISMATCH       -> brake_point_differs
  OVER_BRAKING_PEAK            -> over_braking
  ENTRY_OVER_SLOW              -> entry_too_slow
  EXIT_SPEED_LEGACY            -> exit_slow_no_brake
  POOR_TRAIL_BRAKING           -> no_trail_braking
  LATE_ROTATION                -> rotation_after_vmin
  STEER_EFFICIENCY             -> excess_steering
  AGGRESSIVE_UNSTABLE_BRAKE    -> brake_application_abrupt
  LATE_THROTTLE_LOW_SPEED      -> throttle_hesitation_slow_exit
  THROTTLE_STEER_CONFLICT      -> throttle_with_steering
  LOW_BRAKE_EFFICIENCY         -> None
  ROTATION_INEFFICIENT         -> None


## `BEHAVIOR_CATEGORY` — agrupamento visual do checklist (03)

Reaproveita a `category` que cada regra já tem em `DRIVING_RULES` (`RULE_CAT`) em vez de
inventar um agrupamento novo -- assim os grupos exibidos no instrumento de rotulagem
batem com a taxonomia que o próprio motor de regras já usa. `EXIT_SPEED_LEGACY`
(categoria `"Legacy"`) fica sozinha no grupo "Legado": ela dispara quando o piloto de
teste **já entra no setor mais devagar que a referência** (`SpeedStartDiff < LEGACY_DV`)
e não está freando mais para compensar (`BrakePeak < LEGACY_BRAKE_GATE`) -- ou seja, a
perda é herdada de antes do setor, não causada dentro dele.

In [7]:
BEHAVIOR_CATEGORY = {beh: RULE_CAT[rid] for rid, beh in RULE_BEHAVIOR.items() if beh is not None}

GROUP_LABEL_PT = {
    "Braking":  "Frenagem",
    "Rotation": "Curva / Rotação",
    "Traction": "Aceleração / Tração",
    "Legacy":   "Legado (perda herdada de antes do setor)",
}
GROUP_ORDER = ["Braking", "Rotation", "Traction", "Legacy"]

assert set(BEHAVIOR_CATEGORY.values()) <= set(GROUP_LABEL_PT), \
    "GROUP_LABEL_PT must cover every category among labeling behaviors"

print("Behavior groups for the labeling checklist (reused from DRIVING_RULES categories):")
for group in GROUP_ORDER:
    members = [b for b, g in BEHAVIOR_CATEGORY.items() if g == group]
    if members:
        print(f"  {GROUP_LABEL_PT[group]:45s} ({group}): {members}")

Behavior groups for the labeling checklist (reused from DRIVING_RULES categories):
  Frenagem                                      (Braking): ['brake_point_differs', 'over_braking', 'entry_too_slow', 'brake_application_abrupt']
  Curva / Rotação                               (Rotation): ['no_trail_braking', 'rotation_after_vmin', 'excess_steering']
  Aceleração / Tração                           (Traction): ['throttle_hesitation_slow_exit', 'throttle_with_steering']
  Legado (perda herdada de antes do setor)      (Legacy): ['exit_slow_no_brake']


## Motor de cache de telemetria (dois estágios)

`load_best_lap_interp` carrega um stint, mantém a volta válida mais rápida e alinha os
canais por `LapDistPct`. `interp_cache` é persistido em disco; execuções seguintes
carregam do `.pkl` sem tocar nos `.ibt`.

`RUN_FULL_BUILD` controla o escopo: `False` (padrão desta execução, por decisão do
usuário) roda apenas um subconjunto real pequeno para validar o pipeline (SMOKE TEST);
`True` roda o `PLAN` completo (~54 arquivos, 6 pilotos × 2 pistas, 10-20 min estimados) e
é o que deve ser usado antes de rodar `01`-`04` para valer.

In [8]:
grid = np.linspace(0.0, 1.0, BASE_GRID_LEN)

INTERP_CACHE_PATH = CACHE_DIR / "interp_cache.pkl"

interp_cache = {}
if INTERP_CACHE_PATH.exists():
    interp_cache = pd.read_pickle(INTERP_CACHE_PATH)
    print(f"Loaded interp_cache from disk: {len(interp_cache)} laps, "
          f"no .ibt touched. Delete {INTERP_CACHE_PATH.name} to force a rebuild.")
else:
    print("No cache on disk yet -- laps will be loaded from .ibt as requested below.")

_scale_checked = False


def load_best_lap_interp(track, driver, stint):
    """Load a stint, keep the fastest valid lap, return aligned channels. Memoised in interp_cache."""
    global _scale_checked
    key = (track, driver, stint)
    if key in interp_cache:
        return interp_cache[key]
    files  = [str(p) for p in resolve_stint_files(track, driver, stint)]
    raw    = load_stint(files)
    df     = basic_clean_and_units(raw)
    lap_df = build_lap_validity_table(df)
    valid  = lap_df[lap_df["Valid"]]
    if valid.empty:
        raise RuntimeError(f"No valid lap: {track}/{driver}/{stint}")
    best_row = valid.sort_values("LapTime_s").iloc[0]
    g = df[df["Lap"] == int(best_row["Lap"])]
    interp = align_lap_by_dist(g, grid)
    if not _scale_checked:
        assert_pedal_scale(interp, f"{track}/{driver}/{stint}")
        print(f"[SCALE] pedal channels confirmed on a 0-100 scale "
              f"(brake max = {np.max(interp['brake']):.1f} %)")
        _scale_checked = True
    interp_cache[key] = (interp, float(best_row["LapTime_s"]), int(best_row["Lap"]))
    return interp_cache[key]


# --- Comparison plan (copied from v3) ------------------------------------------
PLAN = []   # (track, TrackShort, label, stint, ref_driver, ref_stint, test_driver)
for track, cfg in DATASETS.items():
    for comp in COMPARISONS:
        if comp["test"] not in cfg["sessions"] or comp["ref"] not in cfg["sessions"]:
            print(f"[SKIP] {track}: {comp['label']} - driver not present on this track")
            continue
        for stint in sorted(k for k in cfg["sessions"][comp["test"]] if k not in EXCLUDE_STINTS):
            PLAN.append((track, TRACK_SHORT[track], comp["label"], stint,
                         comp["ref"], comp["ref_stint"], comp["test"]))

print(f"\nPLAN has {len(PLAN)} comparison-stints (full design).")

RUN_FULL_BUILD = False   # set True to build the complete telemetry cache (10-20 min)

if RUN_FULL_BUILD:
    scope = PLAN
    print("RUN_FULL_BUILD = True -> loading the complete comparison design.")
else:
    scope = [p for p in PLAN if p[0] == "charlotte_roval_2025" and p[2] == "C vs. B"
             and p[3] == "stint_1"]
    print("RUN_FULL_BUILD = False -> SMOKE TEST scope only "
          "(Tomaz/stint_ref, Tomaz/stint_1, Morsinaldo/stint_1, charlotte_roval_2025). "
          "Run with RUN_FULL_BUILD = True to build the real cache before running 01-04.")

lap_records = []
for track, tshort, label, stint, ref_d, ref_s, test_d in scope:
    for drv, stn in ((ref_d, ref_s), (test_d, stint)):
        try:
            load_best_lap_interp(track, drv, stn)
        except Exception as e:
            print(f"[SKIP] {track}/{drv}/{stn} -> {e}")
    try:
        lt, lap = interp_cache[(track, test_d, stint)][1:]
        lap_records.append({"Track": tshort, "Comparison": label, "Stint": stint,
                            "BestLap": lap, "LapTime_s": lt})
    except (KeyError, ValueError):
        pass

# Also load Tomaz/stint_1 explicitly so the self-comparison smoke test has data.
if not RUN_FULL_BUILD:
    try:
        load_best_lap_interp("charlotte_roval_2025", "Tomaz", "stint_1")
    except Exception as e:
        print(f"[SKIP] self smoke stint -> {e}")

df_laps = pd.DataFrame(lap_records)
print(f"\n{len(interp_cache)} laps cached in memory, {len(scope)} comparison-stints in scope.")
pd.to_pickle(interp_cache, INTERP_CACHE_PATH)
print(f"interp_cache persisted -> {INTERP_CACHE_PATH}")

Loaded interp_cache from disk: 3 laps, no .ibt touched. Delete interp_cache.pkl to force a rebuild.

PLAN has 50 comparison-stints (full design).
RUN_FULL_BUILD = False -> SMOKE TEST scope only (Tomaz/stint_ref, Tomaz/stint_1, Morsinaldo/stint_1, charlotte_roval_2025). Run with RUN_FULL_BUILD = True to build the real cache before running 01-04.

3 laps cached in memory, 1 comparison-stints in scope.
interp_cache persisted -> C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\cache\interp_cache.pkl


In [9]:
# --- Descriptor cache and pair cache (copied from v3) --------------------------
def build_heur_cache(dp):
    """(TrackShort, label, stint) -> (heur_ref, heur_test) under descriptor params dp."""
    out, ref_memo = {}, {}
    for track, tshort, label, stint, ref_d, ref_s, test_d in PLAN:
        if (track, ref_d, ref_s) not in interp_cache or (track, test_d, stint) not in interp_cache:
            continue
        edges = TRACK_CONFIGS[track]["custom_edges"]
        rkey = (track, ref_d, ref_s)
        if rkey not in ref_memo:
            ref_memo[rkey] = compute_sector_heuristics(interp_cache[rkey][0], edges, dp)
        heur_t = compute_sector_heuristics(interp_cache[(track, test_d, stint)][0], edges, dp)
        out[(tshort, label, stint)] = (ref_memo[rkey], heur_t)
    return out


def build_pair_cache(heur_cache):
    """Merge ref/test descriptors per sector -> list of (sector, A, B, delta_t)."""
    pair = {}
    for key, (hr, ht) in heur_cache.items():
        m = pd.merge(hr, ht, on="Sector", suffixes=("_A", "_B"))
        rows = []
        for _, row in m.iterrows():
            A = {k[:-2]: v for k, v in row.items() if k.endswith("_A")}
            B = {k[:-2]: v for k, v in row.items() if k.endswith("_B")}
            B["SpeedStartDiff"] = row.get("SpeedStart_B", 0) - row.get("SpeedStart_A", 0)
            dts = float(row.get("SectorTime_s_B", np.nan)) - float(row.get("SectorTime_s_A", np.nan))
            rows.append((int(row["Sector"]), A, B, dts))
        pair[key] = rows
    return pair


def evaluate_rules(pair_cache, th):
    """Activation table under thresholds th. Rules read the global TH at call time."""
    global TH
    th_saved, TH = TH, th
    try:
        rows = []
        for (track, label, stint), pairs in pair_cache.items():
            for sector, A, B, _ in pairs:
                for rule in DRIVING_RULES:
                    try:
                        fired = bool(rule["condition"](A, B))
                    except Exception:
                        fired = False
                    rows.append({"Track": track, "Comparison": label, "Stint": stint,
                                 "Sector": sector, "Rule_ID": rule["id"],
                                 "Category": rule["category"], "Fired": int(fired)})
        return pd.DataFrame(rows)
    finally:
        TH = th_saved


def pair_cache_rows(pair_cache):
    """Flatten a pair_cache dict into a tidy DataFrame with columns
    Track, Comparison, Stint, Sector, dts (sector time delta, test - ref, seconds).
    Specific to this POC -- pair_cache itself stays a dict of tuples as in v3."""
    rows = []
    for (track, label, stint), pairs in pair_cache.items():
        for sector, A, B, dts in pairs:
            rows.append({"Track": track, "Comparison": label, "Stint": stint,
                         "Sector": sector, "dts": dts})
    return pd.DataFrame(rows)


def self_pair_cache(dp):
    """B vs. B (self) pairs: Tomaz/stint_ref as both branches of DRIVER_ALIAS['B'].
    Copied from v3 section 10 (negative control), generalised to any descriptor spec."""
    out = {}
    for track, cfg in DATASETS.items():
        if SELF["test"] not in cfg["sessions"] or SELF["ref"] not in cfg["sessions"]:
            continue
        edges = TRACK_CONFIGS[track]["custom_edges"]
        try:
            interp_ref = load_best_lap_interp(track, SELF["ref"], SELF["ref_stint"])[0]
        except Exception as e:
            print(f"[SKIP] {track} self ref -> {e}")
            continue
        heur_ref = compute_sector_heuristics(interp_ref, edges, dp)
        for stint in sorted(k for k in cfg["sessions"][SELF["test"]] if k not in EXCLUDE_STINTS):
            if (track, SELF["test"], stint) not in interp_cache:
                continue
            interp_t = interp_cache[(track, SELF["test"], stint)][0]
            heur_t = compute_sector_heuristics(interp_t, edges, dp)
            out[(TRACK_SHORT[track], SELF["label"], stint)] = (heur_ref, heur_t)
    return build_pair_cache(out)


heur_cache = build_heur_cache(DESC_SPEC)
pair_cache = build_pair_cache(heur_cache)
df_sector  = evaluate_rules(pair_cache, thresholds_at())

pair_cache_self = self_pair_cache(DESC_SPEC)
df_sector_self  = evaluate_rules(pair_cache_self, thresholds_at())

print(f"pair_cache      : {len(pair_cache)} comparison-stints, "
      f"{sum(len(v) for v in pair_cache.values())} sector pairs.")
print(f"pair_cache_self : {len(pair_cache_self)} comparison-stints, "
      f"{sum(len(v) for v in pair_cache_self.values())} sector pairs.")
print(f"Total activations (cross-driver, current scope): {int(df_sector['Fired'].sum())}")
print(f"Total activations (self, current scope)        : {int(df_sector_self['Fired'].sum())}")

pd.to_pickle(pair_cache, CACHE_DIR / "pair_cache.pkl")
pd.to_pickle(pair_cache_self, CACHE_DIR / "pair_cache_self.pkl")
df_sector.to_parquet(CACHE_DIR / "df_sector.parquet", index=False)
df_sector_self.to_parquet(CACHE_DIR / "df_sector_self.parquet", index=False)
print(f"\nCache persisted under {CACHE_DIR}")

    [1/1] Loading toyotagr86_summit summit raceway 2026-01-27 22-21-35(Tomaz-stint_ref).ibt … 

OK (60,130 samples, laps 0–12)
──────────────────────────────────────────────────────────────
  Lap Validity Report  (13 laps total)
──────────────────────────────────────────────────────────────
  ✅ Valid   : 9 laps
  🏆 Fastest : Lap 8  (01:19.767)
  ❌ Invalid : 4 laps
     Lap   0  16:39.700  → LapTime=999.7s | GPS=27.0%<100%
     Lap   1  01:22.067  → IQR_outlier
     Lap   3  01:22.783  → IQR_outlier
     Lap  12  01:22.633  → CompletedPct=0.938<0.995 | GPS=94.0%<100%
──────────────────────────────────────────────────────────────
[SCALE] pedal channels confirmed on a 0-100 scale (brake max = 78.5 %)
pair_cache      : 1 comparison-stints, 18 sector pairs.
pair_cache_self : 1 comparison-stints, 18 sector pairs.
Total activations (cross-driver, current scope): 47
Total activations (self, current scope)        : 31

Cache persisted under C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\cache


## Auxiliares longitudinais (copiados de `heuristics_activation_pipeline_v2.ipynb`)

`stint_slopes` e `pattern_stats` são usados em `01_gap_closure.ipynb` (varredura de
descritores) para resumir o padrão longitudinal (feedback vs. controle) de cada cenário.

In [10]:
import re as _re


def stint_num(s):
    m = _re.search(r"(\d+)", str(s))
    return int(m.group(1)) if m else 0


def stint_slopes(df_act):
    """Least-squares slope of total activations vs stint index, per (track, comparison)."""
    tot = df_act.groupby(["Track", "Comparison", "Stint"])["Fired"].sum().reset_index()
    tot["StintN"] = tot["Stint"].map(stint_num)
    out = {}
    for (track, comp), g in tot.groupby(["Track", "Comparison"]):
        g = g.sort_values("StintN")
        if len(g) >= 3:
            out[(track, comp)] = float(np.polyfit(g["StintN"], g["Fired"], 1)[0])
    return out


def pattern_stats(slopes):
    """Continuous summary of the longitudinal pattern."""
    fb = [v for (t, c), v in slopes.items() if c in FEEDBACK_PAIRS]
    ct = [v for (t, c), v in slopes.items() if c == CONTROL_PAIR]
    if not fb or not ct:
        return {"slope_fb": np.nan, "slope_ctrl": np.nan,
                "margin": np.nan, "preserved": None}
    slope_fb, slope_ct = float(np.mean(fb)), float(np.mean(ct))
    return {"slope_fb": slope_fb, "slope_ctrl": slope_ct,
            "margin": slope_ct - slope_fb,
            "preserved": bool(slope_fb < 0 and slope_ct > slope_fb)}


def scenario_summary(df_act, base_rank):
    tot  = df_act.groupby("Rule_ID")["Fired"].sum().reindex(RULE_IDS).fillna(0)
    tau  = kendalltau(base_rank, tot.rank(ascending=False))[0]
    stats = pattern_stats(stint_slopes(df_act))
    return {"Total": int(tot.sum()), "KendallTau": float(tau), **stats}, tot


print("stint_slopes / pattern_stats / scenario_summary loaded.")

stint_slopes / pattern_stats / scenario_summary loaded.


## Sanity check final

Compara o total de ativações de `df_sector` (baseline, `thresholds_at()` sem fatores)
com `heuristic_activations_long.csv` do v3 — **só é uma comparação exata quando o cache
cobre o `PLAN` completo** (`RUN_FULL_BUILD = True`). Com `RUN_FULL_BUILD = False`
(escopo desta execução), a célula compara o subconjunto correspondente e reporta que a
comparação total fica pendente.

In [11]:
V3_CSV = PROJECT_ROOT / "img" / "batch_heuristics_v3" / "heuristic_activations_long.csv"

if not V3_CSV.exists():
    print(f"[WARN] {V3_CSV} not found -- sanity check skipped.")
elif not RUN_FULL_BUILD:
    print("RUN_FULL_BUILD = False: cache only covers a smoke-test subset, "
          "so an exact total-activation match against the full v3 CSV is not expected.")
    df_v3 = pd.read_csv(V3_CSV)
    scope_keys = {(tshort, label, stint) for _, tshort, label, stint, *_ in scope}
    v3_scope = df_v3[df_v3.apply(lambda r: (r["Track"], r["Comparison"], r["Stint"]) in scope_keys, axis=1)]
    v3_total = int(v3_scope["Fired"].sum())
    poc_total = int(df_sector["Fired"].sum())
    print(f"v3 total (same scope)  : {v3_total}")
    print(f"POC total (this scope) : {poc_total}")
    print("MATCH" if v3_total == poc_total else "MISMATCH -- investigate before trusting downstream notebooks")
    print("\nRun with RUN_FULL_BUILD = True to compare the complete design against the v3 CSV.")
else:
    df_v3 = pd.read_csv(V3_CSV)
    v3_total  = int(df_v3["Fired"].sum())
    poc_total = int(df_sector["Fired"].sum())
    print(f"v3 total (full design)  : {v3_total}")
    print(f"POC total (full design) : {poc_total}")
    print("MATCH -- POC baseline reproduces v3 exactly." if v3_total == poc_total
          else "MISMATCH -- investigate before trusting downstream notebooks")

RUN_FULL_BUILD = False: cache only covers a smoke-test subset, so an exact total-activation match against the full v3 CSV is not expected.
v3 total (same scope)  : 46
POC total (this scope) : 47
MISMATCH -- investigate before trusting downstream notebooks

Run with RUN_FULL_BUILD = True to compare the complete design against the v3 CSV.
